In [2]:
import sys
from pathlib import Path
import pandas as pd

sys.path.insert(0, str(Path.cwd().parent))   # so config.py (one folder up) can be imported
from config import get_engine

df = pd.read_sql("SELECT * FROM encounters_clean", get_engine())
print(df.shape)

(99340, 29)


In [3]:
target = "readmitted_30"
drop_cols = ["encounter_id", "patient_nbr", "readmitted", "readmit_status",
             "readmitted_30", "a1c_tested"]

X = df.drop(columns=drop_cols)
y = df[target]
groups = df["patient_nbr"]          # used to split by patient

num_cols = X.select_dtypes(include="number").columns.tolist()
cat_cols = X.select_dtypes(exclude="number").columns.tolist()
X[cat_cols] = X[cat_cols].fillna("Unknown")

print("Number columns:", len(num_cols), "| Text columns:", len(cat_cols))
print(f"Readmitted within 30 days: {y.mean():.1%}")

Number columns: 9 | Text columns: 14
Readmitted within 30 days: 11.4%


In [4]:
from sklearn.model_selection import GroupShuffleSplit

splitter = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
train_idx, test_idx = next(splitter.split(X, y, groups))

X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

print("Train:", X_train.shape, "| Test:", X_test.shape)
print("Patients in both train and test:",
      len(set(groups.iloc[train_idx]) & set(groups.iloc[test_idx])))   # should be 0

Train: (79567, 23) | Test: (19773, 23)
Patients in both train and test: 0


## 4. Build and train the models
Each model is a **Pipeline**: first prepare the data, then train.
- `StandardScaler` puts number columns on the same scale.
- `OneHotEncoder` turns text columns into 0/1 number columns.

We train two models on exactly the same data:
1. **Linear Regression**
2. **Logistic Regression**

In [5]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression, LogisticRegression

def make_pipeline(model):
    prep = ColumnTransformer([
        ("num", StandardScaler(), num_cols),
        ("cat", OneHotEncoder(handle_unknown="ignore"), cat_cols),
    ])
    return Pipeline([("prep", prep), ("model", model)])

models = {
    "Linear Regression": make_pipeline(LinearRegression()),
    "Logistic Regression": make_pipeline(LogisticRegression(max_iter=1000)),
}

for name, m in models.items():
    m.fit(X_train, y_train)
    print("Trained:", name)

Trained: Linear Regression
Trained: Logistic Regression


## 5. Compare the models
- **ROC-AUC**: how well the model ranks risky patients above safe ones (0.5 = guessing, 1.0 = perfect).
- **Accuracy**: % of correct predictions. Misleading here, because only about 11% are readmitted.
- **Recall**: of the patients who really came back, how many did we catch? **Most important for a hospital.**
- **Precision**: of the patients we flagged, how many really came back?
- A score of 0.5 or more counts as "will be readmitted".

In [6]:
from sklearn.metrics import (roc_auc_score, accuracy_score, recall_score,
                             precision_score, f1_score)

rows = []
scores = {}
for name, m in models.items():
    if name == "Linear Regression":
        score = m.predict(X_test)                 # any number
    else:
        score = m.predict_proba(X_test)[:, 1]     # probability 0 to 1
    scores[name] = score
    pred = (score >= 0.5).astype(int)

    rows.append({
        "Model": name,
        "ROC-AUC": roc_auc_score(y_test, score),
        "Accuracy": accuracy_score(y_test, pred),
        "Recall": recall_score(y_test, pred),
        "Precision": precision_score(y_test, pred, zero_division=0),
        "F1": f1_score(y_test, pred),
        "Patients flagged": int(pred.sum()),
        "Outputs below 0": int((score < 0).sum()),
        "Outputs above 1": int((score > 1).sum()),
    })

results = pd.DataFrame(rows).set_index("Model").round(3)
print("Real readmissions in test set:", int(y_test.sum()))
print("Accuracy of always saying 'No':", round(1 - y_test.mean(), 3))
results

Real readmissions in test set: 2244
Accuracy of always saying 'No': 0.887


,ROC-AUC,Accuracy,Recall,Precision,F1,Patients flagged,Outputs below 0,Outputs above 1
Model,,,,,,,,
Linear Regression,0.663,0.887,0.011,0.714,0.022,35,64,0
Logistic Regression,0.661,0.888,0.020,0.652,0.039,69,0,0


## 6. What does each model actually output?

In [7]:
import plotly.express as px

plot_df = pd.DataFrame({
    "Linear Regression": scores["Linear Regression"],
    "Logistic Regression": scores["Logistic Regression"],
}).melt(var_name="Model", value_name="Predicted value")

fig = px.histogram(plot_df, x="Predicted value", color="Model",
                   barmode="overlay", nbins=80,
                   title="What each model outputs (red line = 0)")
fig.add_vline(x=0, line_dash="dash", line_color="red")
fig.show()

## 7. Analysis

1. **ROC-AUC is almost the same** (about 0.663 vs 0.661). Both models rank risky patients above safe ones about equally well. The 0.002 difference is too small to matter.
2. **Accuracy is misleading.** Both models get about 88.7% accuracy, but always saying "No" also gets 88.7%. So accuracy alone can't tell us which model is better.
3. **Linear Regression gives impossible values.** About 65 patients got a prediction below 0. You can't have a negative chance of being readmitted, so these outputs are not real probabilities. Logistic Regression always stays between 0 and 1 (see the chart above).
4. **Logistic Regression catches a few more readmissions.** At the 0.5 cutoff it flagged about 69 patients and had higher recall (about 2.0% vs 1.1%), and its F1 score is also better (about 0.039 vs 0.022).
5. **Logistic Regression is the right tool for the job.** It is designed for yes/no problems and its output can be read directly as risk, for example "this patient has a 30% chance of readmission". Linear Regression is designed for predicting numbers like days in hospital.

## 8. Model choice: Logistic Regression

I compared Linear Regression and Logistic Regression on the same patient-level train/test split. Both ranked patients about equally well (ROC-AUC about 0.66), and both reached about 88.7% accuracy, which is the same as always predicting "No". However, Linear Regression produced impossible values below 0, while Logistic Regression always gives a valid probability between 0 and 1 and caught slightly more readmissions. Because the target is a yes/no answer, I chose **Logistic Regression**.

**Limitations:** ROC-AUC of about 0.66 is moderate, which is normal for this dataset. Both models catch only a small share of readmissions at the 0.5 cutoff, because only about 11% of patients are readmitted. 